<a href="https://colab.research.google.com/github/Mogi21/ml-algorithms-from-scratch/blob/main/03_polynomial_lasso_and_nested_cv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from itertools import combinations_with_replacement

class PolynomialFeaturesFromScratch:
    """Generates polynomial and interaction features up to a specified degree."""
    def __init__(self, degree=2):
        self.degree = degree

    def fit_transform(self, X):
        n_samples, n_features = X.shape
        features = [np.ones((n_samples, 1))] # Bias/Intercept term

        for d in range(1, self.degree + 1):
            for combos in combinations_with_replacement(range(n_features), d):
                new_feature = np.prod(X[:, combos], axis=1, keepdims=True)
                features.append(new_feature)

        return np.hstack(features)


class LassoCoordinateDescent:
    """
    Lasso Regression with L1 regularization using Coordinate Descent.
    Applies soft-thresholding to induce sparsity (feature selection).
    """
    def __init__(self, alpha=1.0, max_iter=1000, tol=1e-4):
        self.alpha = alpha
        self.max_iter = max_iter
        self.tol = tol
        self.weights = None

    def _soft_thresholding(self, rho, alpha):
        """Applies the soft thresholding operator."""
        if rho < -alpha:
            return rho + alpha
        elif rho > alpha:
            return rho - alpha
        else:
            return 0.0

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)

        for _ in range(self.max_iter):
            weights_old = self.weights.copy()

            for j in range(n_features):
                # Exclude the current feature's weight from the prediction
                y_pred_without_j = X @ self.weights - X[:, j] * self.weights[j]
                residual = y - y_pred_without_j

                rho = X[:, j] @ residual
                z = np.sum(X[:, j]**2)

                if z == 0:
                    self.weights[j] = 0
                else:
                    # Do not regularize the intercept (column 0)
                    if j == 0:
                        self.weights[j] = rho / z
                    else:
                        self.weights[j] = self._soft_thresholding(rho, self.alpha * n_samples) / z

            # Check for convergence
            if np.max(np.abs(self.weights - weights_old)) < self.tol:
                break

    def predict(self, X):
        return X @ self.weights

In [2]:
def k_fold_split(n_samples, n_splits=5, random_state=None):
    """Generates train/test indices for k-fold cross-validation."""
    rng = np.random.default_rng(random_state)
    indices = rng.permutation(n_samples)
    fold_sizes = np.full(n_splits, n_samples // n_splits, dtype=int)
    fold_sizes[:n_samples % n_splits] += 1

    current = 0
    folds = []
    for fold_size in fold_sizes:
        start, stop = current, current + fold_size
        test_idx = indices[start:stop]
        train_idx = np.concatenate((indices[:start], indices[stop:]))
        folds.append((train_idx, test_idx))
        current = stop
    return folds

def nested_cross_validation(X, y, alphas, degree=2, outer_cv=5, inner_cv=5):
    """
    Double-loop validation pipeline.
    Inner loop: Tunes the L1 penalty (alpha) for Lasso.
    Outer loop: Evaluates the generalized performance of the unbiased model.
    """
    outer_folds = k_fold_split(len(X), n_splits=outer_cv, random_state=42)
    outer_errors = []
    best_alphas = []

    # Expand features once before splitting to save computation
    poly = PolynomialFeaturesFromScratch(degree=degree)
    X_poly = poly.fit_transform(X)

    for i, (train_idx, test_idx) in enumerate(outer_folds):
        X_train_outer, y_train_outer = X_poly[train_idx], y[train_idx]
        X_test_outer, y_test_outer = X_poly[test_idx], y[test_idx]

        inner_folds = k_fold_split(len(X_train_outer), n_splits=inner_cv, random_state=42)

        # --- Inner Loop: Hyperparameter Tuning ---
        best_alpha = None
        best_inner_error = float('inf')

        for alpha in alphas:
            inner_errors = []
            for inner_train_idx, inner_val_idx in inner_folds:
                X_train_inner, y_train_inner = X_train_outer[inner_train_idx], y_train_outer[inner_train_idx]
                X_val_inner, y_val_inner = X_train_outer[inner_val_idx], y_train_outer[inner_val_idx]

                model = LassoCoordinateDescent(alpha=alpha, max_iter=500)
                model.fit(X_train_inner, y_train_inner)
                y_pred_inner = model.predict(X_val_inner)

                mse = np.mean((y_val_inner - y_pred_inner)**2)
                inner_errors.append(mse)

            mean_inner_error = np.mean(inner_errors)
            if mean_inner_error < best_inner_error:
                best_inner_error = mean_inner_error
                best_alpha = alpha

        best_alphas.append(best_alpha)

        # --- Outer Loop: Generalization Evaluation ---
        final_model = LassoCoordinateDescent(alpha=best_alpha, max_iter=1000)
        final_model.fit(X_train_outer, y_train_outer)
        y_pred_outer = final_model.predict(X_test_outer)

        outer_mse = np.mean((y_test_outer - y_pred_outer)**2)
        outer_errors.append(outer_mse)

        print(f"Outer Fold {i+1}: Selected Alpha = {best_alpha:.4f} | Test MSE = {outer_mse:.4f}")

    print(f"\nFinal Estimated Generalization Error (MSE): {np.mean(outer_errors):.4f} ± {np.std(outer_errors):.4f}")
    return outer_errors, best_alphas

In [3]:
from sklearn.datasets import make_regression
from sklearn.preprocessing import StandardScaler

print("[ Generating Non-Linear Data for Polynomial Lasso ]")
# Generate data that requires polynomial features to solve effectively
X_raw, y_raw = make_regression(n_samples=200, n_features=3, noise=1.5, random_state=42)

# Inject a non-linear relationship (squaring the first feature) to prove the polynomial expansion works
y = y_raw + 2.5 * (X_raw[:, 0] ** 2)

# Standardize features (crucial for L1 regularization scaling)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print("\n[ Running Nested Cross-Validation Pipeline (Outer=5, Inner=5) ]")
# Define hyperparameter grid for L1 penalty
alpha_grid = [0.01, 0.1, 0.5, 1.0, 2.0]

outer_errors, chosen_alphas = nested_cross_validation(
    X=X_scaled,
    y=y,
    alphas=alpha_grid,
    degree=2,
    outer_cv=5,
    inner_cv=5
)

[ Generating Non-Linear Data for Polynomial Lasso ]

[ Running Nested Cross-Validation Pipeline (Outer=5, Inner=5) ]
Outer Fold 1: Selected Alpha = 0.0100 | Test MSE = 1.8961
Outer Fold 2: Selected Alpha = 0.0100 | Test MSE = 2.8029
Outer Fold 3: Selected Alpha = 0.1000 | Test MSE = 2.0717
Outer Fold 4: Selected Alpha = 0.0100 | Test MSE = 2.9308
Outer Fold 5: Selected Alpha = 0.1000 | Test MSE = 3.2081

Final Estimated Generalization Error (MSE): 2.5819 ± 0.5086
